# Creating Machine Learning Ready Datasets

The purpose of this notebook is to merge the macroeconomic dataset and the price data. We need to forward fill the data that comes infrequently. We then need to calculate the forward looking log retruns (by one quarter) and annualize it, and then calculate the forward looking volatility and annualize that. These vectors will be our labels.

After that, I want to create a second CSV that uses stationary data (differenced) instead of the raw data to see whether or not this imporves the performance of our LSTM. This dataset will be more similar to how financial analysts and economoists look at data, however, it is possible that deep learning will find similar patterns in the raw data because of the nature of deep learning. We will compare model performance with the two datasets.

## Import Libraries
Let's create a section for all of the libraries that will be necessary for this manipulation.

In [59]:
# Manipulation libraries  
import numpy as np
import pandas as pd
import altair as alt

# Deactivate the max rows and columns limit for Altair
alt.data_transformers.disable_max_rows()


DataTransformerRegistry.enable('default')

# Import the datasets
Now, we can pull in both of the CSV files that we will merge together.

In [38]:
macro_df = pd.read_csv('fred_data.csv')
price_df = pd.read_csv('yfinance_data.csv')
print(f"Macro DataFrame shape: {macro_df.shape}, Price DataFrame shape: {price_df.shape}")

Macro DataFrame shape: (20125, 26), Price DataFrame shape: (25009, 38)


Now, let's take a look at the dataframes to make sure we can merge on the 'date'.

In [39]:
macro_df.head()

,date,nominal_GDP,real_GDP,debt_to_GDP,debt_interest,consumer_price_index,core_PCI,personal_consumption_expenditure,core_PCE,producer_price_index,...,female_unemployment_rate,average_duration_of_unemployment,one_month_yield,three_month_yield,six_month_yield,one_year_yield,two_year_yield,five_year_yield,ten_year_yield,thirty_year_yield
0,1946-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1946-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1946-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1946-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1947-01-01,243.164,2182.681,NaN,5.352,21.48,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
price_df.head()

,date,BIL,BND,GLD,HYG_x,IEF,IWM,LQD_x,QQQ,SPY,...,NG=F,SHY,TLT_y,^FTSE,^GSPC,^HSI,^MOVE,^TNX,^VIX,^VXN
0,1927-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.660000,NaN,NaN,NaN,NaN,NaN
1,1928-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.760000,NaN,NaN,NaN,NaN,NaN
2,1928-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.719999,NaN,NaN,NaN,NaN,NaN
3,1928-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.549999,NaN,NaN,NaN,NaN,NaN
4,1928-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.660000,NaN,NaN,NaN,NaN,NaN


Great, so this should be a simple horizontal merge of the dataframes on the 'date' column

In [41]:
merged_df = pd.merge(macro_df, price_df, on='date', how='outer')
merged_df.shape

(28526, 63)

Let's do a quick sanity check to see the minimum and maximum date.

In [42]:
min, max = merged_df['date'].min(), merged_df['date'].max()
print(f"Date range: {min} to {max}")

Date range: 1927-12-30 to 2026-03-11


Alright, this looks great. Let's pull all of the column names into a list so we can work with them a little easier.

In [43]:
independent_columns = merged_df.columns.tolist()
print(f"Columns in merged DataFrame: {independent_columns}")

Columns in merged DataFrame: ['date', 'nominal_GDP', 'real_GDP', 'debt_to_GDP', 'debt_interest', 'consumer_price_index', 'core_PCI', 'personal_consumption_expenditure', 'core_PCE', 'producer_price_index', 'unemployment_rate', 'initial_jobless_claims', 'continued_jobless_claims', 'teenager_unemployment_rate', 'adult_unemployment_rate', 'male_unemployment_rate', 'female_unemployment_rate', 'average_duration_of_unemployment', 'one_month_yield', 'three_month_yield', 'six_month_yield', 'one_year_yield', 'two_year_yield', 'five_year_yield', 'ten_year_yield', 'thirty_year_yield', 'BIL', 'BND', 'GLD', 'HYG_x', 'IEF', 'IWM', 'LQD_x', 'QQQ', 'SPY', 'TIP', 'TLT_x', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY', 'CL=F', 'DX-Y.NYB', 'GC=F', 'HG=F', 'HYG_y', 'LQD_y', 'NG=F', 'SHY', 'TLT_y', '^FTSE', '^GSPC', '^HSI', '^MOVE', '^TNX', '^VIX', '^VXN']


Interestingly, we can see that we had some duplicate data (i.e. TLT) when we merged our data in the previous CSVs. So, let's loop through and keep only one of them.

In [44]:
# Drop _y duplicates and rename _x back to original
x_cols = [col for col in merged_df.columns if col.endswith('_x')]

for col in x_cols:
    base = col[:-2]  # strip '_x'
    merged_df.drop(columns=[f'{base}_y'], inplace=True)
    merged_df.rename(columns={col: base}, inplace=True)

Now, let's forward fill the data so that less frequent data is filled into the dataframe so we have complete vectors at each timeframe. First, let's get an idea of the number of NaN's in each column, then forward fill, then double check to see if it is filling what we expect (like GDP)

In [45]:
 merged_df.isna().sum()

date                                    0
nominal_GDP                         28210
real_GDP                            28210
debt_to_GDP                         28287
debt_interest                       28210
consumer_price_index                27578
core_PCI                            27698
personal_consumption_expenditure    27722
core_PCE                            27722
producer_price_index                28331
unemployment_rate                   27590
initial_jobless_claims              25440
continued_jobless_claims            25441
teenager_unemployment_rate          27590
adult_unemployment_rate             27590
male_unemployment_rate              27590
female_unemployment_rate            27590
average_duration_of_unemployment    27590
one_month_yield                     22383
three_month_yield                   17407
six_month_yield                     17407
one_year_yield                      12505
two_year_yield                      16097
five_year_yield                   

In [46]:
filled_df = merged_df.ffill()


In [47]:
filled_df.isna().sum()

date                                    0
nominal_GDP                          4746
real_GDP                             4746
debt_to_GDP                          9633
debt_interest                        4746
consumer_price_index                 4746
core_PCI                             7294
personal_consumption_expenditure     7808
core_PCE                             7808
producer_price_index                23379
unemployment_rate                    5002
initial_jobless_claims               9902
continued_jobless_claims             9902
teenager_unemployment_rate           5002
adult_unemployment_rate              5002
male_unemployment_rate               5002
female_unemployment_rate             5002
average_duration_of_unemployment     5002
one_month_yield                     20781
three_month_yield                   14512
six_month_yield                     14512
one_year_yield                       8576
two_year_yield                      12859
five_year_yield                   

Alright, this looks pretty good. The first 63 columns are our independent variables at the moment. Let's start building the log returns labels. We will need log returns of all of the potential ETF's that we will be using in our portfolio generator. So let's create those.

In [50]:
target_variables = ['XLF','XLK','XLU','XLV','XLE','XLI','XLB','XLP','XLY','XLRE','BIL','IEF','TLT','LQD','HYG','TIP','GLD']

Now that we have our target variables, let's iterate through and create the new columns for our target labels. I am going to append '_target' on the end so that we remember to never use these as features to avoid data leakage.

The first thing we are doing is calculating the log return that is a quarter forward looking (63 days assuming 252 trading days). We will then multiply this by 4 to annualize it.

In [51]:
for ticker in target_variables:
    filled_df[f'{ticker}_logreturn_target'] = np.log(filled_df[ticker].shift(-63)/filled_df[ticker]) * 4

log_return_columns = [f'{ticker}_logreturn_target' for ticker in target_variables]
log_return_columns

['XLF_logreturn_target',
 'XLK_logreturn_target',
 'XLU_logreturn_target',
 'XLV_logreturn_target',
 'XLE_logreturn_target',
 'XLI_logreturn_target',
 'XLB_logreturn_target',
 'XLP_logreturn_target',
 'XLY_logreturn_target',
 'XLRE_logreturn_target',
 'BIL_logreturn_target',
 'IEF_logreturn_target',
 'TLT_logreturn_target',
 'LQD_logreturn_target',
 'HYG_logreturn_target',
 'TIP_logreturn_target',
 'GLD_logreturn_target']

Alright perfect. Now, let's build all of the annualized volatility columns.

In [52]:
for ticker in target_variables:
    filled_df[f'{ticker}_volatility_target'] = (
        np.log(filled_df[ticker] / filled_df[ticker].shift(1))
          .rolling(63)
          .std()
        * np.sqrt(252)
    )

volatility_columns = [f'{ticker}_volatility_target' for ticker in target_variables]
volatility_columns

['XLF_volatility_target',
 'XLK_volatility_target',
 'XLU_volatility_target',
 'XLV_volatility_target',
 'XLE_volatility_target',
 'XLI_volatility_target',
 'XLB_volatility_target',
 'XLP_volatility_target',
 'XLY_volatility_target',
 'XLRE_volatility_target',
 'BIL_volatility_target',
 'IEF_volatility_target',
 'TLT_volatility_target',
 'LQD_volatility_target',
 'HYG_volatility_target',
 'TIP_volatility_target',
 'GLD_volatility_target']

Alright, let's export this as a CSV that can now be worked on. It's important to note that this still has a significant number of NaN values and independent variables that we may end up dropping in our modelling but it is a workable dataset to start designing our machine learning pipeline around.

In [56]:
filled_df.to_csv('raw_data_prediction_dataset.csv', index=False)

Let's take a quick look visually to see if these targets make sense.

In [68]:
XLK_return_df = filled_df[['date', 'XLU_logreturn_target', 'XLK_logreturn_target']].copy().dropna()
XLK_return = alt.Chart(XLK_return_df).mark_line().encode(
    x=alt.X('date:T',axis=alt.Axis(title='Date', grid=False)),
    y=alt.Y('XLK_logreturn_target:Q', axis=alt.Axis(title='Log Return', grid=False))
)
XLU_return = alt.Chart(XLK_return_df).mark_line(color='orange').encode(
    x='date:T',
    y='XLU_logreturn_target:Q'
)

(XLK_return + XLU_return).properties(
    title='XLK vs XLU Log Returns', height=400, width=900).configure_view(strokeWidth=0)

alt.LayerChart(...)

In [69]:
XLK_volatility_df = filled_df[['date', 'XLU_volatility_target', 'XLK_volatility_target']].copy().dropna()
XLK_volatility = alt.Chart(XLK_volatility_df).mark_line().encode(
    x=alt.X('date:T',axis=alt.Axis(title='Date', grid=False)),
    y=alt.Y('XLK_volatility_target:Q', axis=alt.Axis(title='Volatility', grid=False))
)
XLU_volatility = alt.Chart(XLK_volatility_df).mark_line(color='orange').encode(
    x='date:T',
    y='XLU_volatility_target:Q'
)

(XLK_volatility + XLU_volatility).properties(
    title='XLK vs XLU Volatilities', height=400, width=900).configure_view(strokeWidth=0)

alt.LayerChart(...)